# Traceprop-LLM — LDS quality parity (workstream C)

**Claim:** last-block attribution (the <1%-overhead, ~100x-cheaper config) matches all-layer / post-hoc attribution quality.

Linear Datamodeling Score (Park et al. 2023): retrain LoRA on M random 50% subsets, correlate `subset_masks @ attribution` with the actual retrained test margins (Spearman), averaged over test examples. Higher = better. The comparison is controlled — same gradient mechanism, only the tracked layers differ (last-block = Traceprop-LL; all-layers = TRAK-style).

GPU runtime. GPT-2 + LoRA sequence classifier on SST-2.

In [ ]:
!pip -q install transformers peft datasets accelerate
!pip -q uninstall -y torchao 2>/dev/null
import torch; print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
import getpass
TOKEN = getpass.getpass('GitHub token: ').strip()
url = f'https://{TOKEN}@github.com/AmitoVrito/Traceprop.git'
!git clone -q {url} /content/Traceprop || (cd /content/Traceprop && git pull -q)
%cd /content/Traceprop
!pip -q install -e .

## LDS: GPT-2 + LoRA on SST-2

Start with 64 subsets (~10-15 min). Bump `--n_subsets` to 128/256 for a tighter final number. `--track 1` = last-block; the script also computes all-layers internally for the parity comparison.

In [ ]:
%cd /content/Traceprop/experiments
!python exp27_lds_quality.py --backend hf --model gpt2 --device cuda \
    --n_train 1000 --n_test 200 --n_subsets 64 --epochs 3 --seq 64 \
    --batch 16 --proj_dim 256 --track 1

## Result

Look for `last_block_dot` vs `all_layers_dot` (and the TRAK variants). If last-block ≈ all-layers and both ≫ random, the cost win of workstreams A/B comes at no quality cost — the paper's spine is complete.

In [ ]:
import json, glob
for p in glob.glob('/content/Traceprop/experiments/results/exp27_hf_*.json'):
    d = json.load(open(p))
    print(d['model'], 'acc', d['target_test_acc'], 'subsets', d['n_subsets'])
    for k, v in d['lds'].items():
        print(f"  {k:<20} {v['mean']:+.4f} ± {v['std']:.4f}")